# repeated_cross_validation

Generated with **Lispyn** — originally created by **Eduardo J. Barrios**.

Source: [Lispyn](https://github.com/edujbarrios/lispyn) (Apache-2.0).

In [ ]:
import hy
hy.macros.require('lispyn.macros', None, target_module_name='repeated_cross_validation', assignments=[['md', 'md']], prefix='')


# Repeated, leakage-safe cross-validation with Lisp macros

Train a fresh native PyTorch classifier for each validation fold.

Reserve the test set for *final* evaluation only, never for model selection.

In [ ]:
import hy
hy.macros.require('lispyn.macros', None, target_module_name='repeated_cross_validation', assignments=[['train-val-test', 'train-val-test'], ['cross-validate', 'cross-validate'], ['train-until', 'train-until'], ['save-cross-validation', 'save-cross-validation']], prefix='')
import torch
features = torch.tensor([[i / 40.0, 1.0] for i in range(40)], dtype=torch.float32)
labels = [0 if i < 20 else 1 for i in range(40)]
dataset = torch.utils.data.TensorDataset(features, torch.tensor(labels, dtype=torch.long))
from lispyn.data import split_dataset_three
partitions = split_dataset_three(dataset, labels=labels, validation_fraction=0.2, test_fraction=0.1, seed=42)
training = partitions[0]
test_data = partitions[2]
train_labels = [labels[index] for index in getattr(training, 'indices')]

def score_fold(fold_training, fold_validation, fold, repeat):
    model = torch.nn.Linear(2, 2)
    from lispyn.workflow import train
    run = train(model, fold_training, patience=2, validation=fold_validation, loss_fn=torch.nn.CrossEntropyLoss(), epochs=4, lr=0.05, batch_size=8, seed=42 + repeat * 100 + fold, metrics='accuracy')
    return run.history['val_accuracy'][-1]
from lispyn.cv_workflows import run_dataset_cross_validation
report = run_dataset_cross_validation(training, score_fold, n_splits=4, n_repeats=2, labels=train_labels, seed=42, metric='val_accuracy')
print('Mean fold accuracy:', report['mean'])
print('Validation-size weighted mean:', report['weighted_mean'])
print('Test examples held out:', len(test_data))
from lispyn.cv_workflows import save_cv_report
save_cv_report(report, 'runs/repeated_cv_report.json')
